# Imports

In [144]:
import math
import numpy as np
import pandas as pd
import plotly.express as px
import pickle

# Data Loading

In [145]:
train_data = pd.read_csv('/kaggle/input/datasets/andonians/random-linear-regression/train.csv')
test_data = pd.read_csv('/kaggle/input/datasets/andonians/random-linear-regression/test.csv')

train_data.head()

,x,y
0,24.0,21.549452
1,50.0,47.464463
2,15.0,17.218656
3,38.0,36.586398
4,87.0,87.288984


 > This cell of train_data.dropna() was added after i went through an error in the data in the later steps. I was getting all cost values as nan . Later in this notebook there is one cell where i check the data for nan values. I manually tested forward pass , backward pass and cost values to find that y_train had ONE NAN VALUE IN ROW 213. so this cell is added to deal with that. After dropping it the model works.

In [146]:
train_data = train_data.dropna()

In [147]:
fig = px.scatter(train_data, x='x', y='y', title='Training Data')
fig.show()

# Data Preprocessing

## Standardize the data
    Standardization is the process of rescaling data so that it has
    a mean of 0 and a standard deviation of 1.
### Why Use Standardization in Machine Learning?
 Our model learns by taking small steps to reduce its error.
If the data is not standardized, some of these steps will be
too large and some too small, making the learning process slow
and unstable. Standardizing the data makes the learning process
smoother and faster.imagine you are comparing the heights of
people measured in centimeters (160, 170, 180) versus their weights
measured in kilograms (60, 70, 80). The numbers look similar here
but in real datasets features can have very different ranges like
0-100 vs 0-100000. Standardization brings everything to the same
scale so no single feature dominates.

### How to Standardize Data

     z = (x - mean) / std
     z is new value
     x is original value
     mean is the mean of all values
     std is standard deviation

In [148]:
X_train = train_data['x'].values.astype(float)
y_train = train_data['y'].values.astype(float)
X_test = test_data['x'].values.astype(float)
y_test = test_data['y'].values.astype(float)
mean = X_train.mean()
std = X_train.std()
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std
print("mean:", mean)
print("std:", std)
#Going by the formula given above

mean: 50.01430615164521
std: 28.933841385275375


## Reshaping data for the correct shape for the model

why cant we make the model without reshaping?

sklearn and numpy expect X to be a 2D array of shape (samples, features).
A 1D array of shape (700,) has no concept of "columns" — the model
doesn't know if it's one feature with 700 samples or 700 features
with 1 sample. Reshaping to (700, 1) makes it unambiguous.

In [149]:
X_train = X_train.reshape(-1, 1)
X_test = X_test.reshape(-1, 1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (699, 1)
y_train shape: (699,)
X_test shape: (300, 1)
y_test shape: (300,)


# Model Implementation

# Linear Regression Model

Linear regression is a fundamental model in machine learning used for predicting a continuous output variable based on input features. The model function for linear regression is represented as:

$$f_{w,b}(x) = wx + b$$

In this equation, $f_{w,b}(x)$ represents the predicted output, $w$ is the weight parameter, $b$ is the bias parameter, and $x$ is the input feature.

## Model Training

To train a linear regression model, we aim to find the best values for the parameters $(w, b)$ that best fit our dataset.

### Forward Pass

The forward pass is a step where we compute the linear regression output for the input data $X$ using the current weights and biases. It's essentially applying our model to the input data.

### Cost Function

The cost function is used to measure how well our model is performing. It quantifies the difference between the predicted values and the actual values in our dataset. The cost function is defined as:

$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m}(f_{w,b}(x^{(i)}) - y^{(i)})^2$$

Here, $J(w, b)$ is the cost, $m$ is the number of training examples, $x^{(i)}$ is the input data for the $i$-th example, $y^{(i)}$ is the actual output for the $i$-th example, and $w$ and $b$ are the weight and bias parameters, respectively.

### Backward Pass (Gradient Computation)

The backward pass computes the gradients of the cost function with respect to the weights and biases. These gradients are crucial for updating the model parameters during training. The gradient formulas are as follows:

$$
\frac{\partial J(w,b)}{\partial b} = \frac{1}{m} \sum_{i=0}^{m-1} (f_{w,b}(X^{(i)}) - y^{(i)})
$$

$$
\frac{\partial J(w,b)}{\partial w} = \frac{1}{m} \sum_{i=0}^{m-1} (f_{w,b}(X^{(i)}) - y^{(i)})X^{(i)}
$$

## Training Process

The training process involves iteratively updating the weights and biases to minimize the cost function. This is typically done through an optimization algorithm like gradient descent. The update equations for parameters are:

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}$$

$$b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

Here, $\alpha$ represents the learning rate, which controls the step size during parameter updates.

By iteratively performing the forward pass, computing the cost, performing the backward pass, and updating the parameters, the model learns to make better predictions and fit the data.


In [150]:
class LinearRegression:

    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
        self.w = None
        self.b = None
        self.cost_history = []

    def initialize_parameters(self, n_features):
        # Start w and b at zero
        self.w = np.zeros(n_features)
        self.b = 0

    def forward(self, X):
        # f(x) = wx + b
        return np.dot(X, self.w) + self.b

    def compute_cost(self, y_pred, y_true):
        m = len(y_true)
        cost = (1 / (2 * m)) * np.sum((y_pred - y_true) ** 2)
        return cost

    def backward(self, X, y_pred, y_true):
        m = len(y_true)
        error = y_pred - y_true
        dw = (1 / m) * np.dot(X.T, error)
        db = (1 / m) * np.sum(error)
        return dw, db

    def fit(self, X, y, iterations, plot_cost=True):
        self.initialize_parameters(X.shape[1])

        for i in range(iterations):
            y_pred = self.forward(X)
            cost = self.compute_cost(y_pred, y)
            self.cost_history.append(cost)
            dw, db = self.backward(X, y_pred, y)

            # Update parameters
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

            if i % 100 == 0:
                print(f"Iteration {i}: Cost = {cost:.4f}")

        if plot_cost:
            fig = px.line(y=self.cost_history, title='Cost over Iterations',
                         labels={'x': 'Iteration', 'y': 'Cost'})
            fig.show()

    def predict(self, X):
        return self.forward(X)

    def save_model(self, filename=None):
        with open(filename, 'wb') as f:
            pickle.dump(self, f)
        print(f"Model saved to {filename}")

    @classmethod
    def load_model(cls, filename):
        with open(filename, 'rb') as f:
            return pickle.load(f)

In [151]:
# Manually test forward pass
lr = LinearRegression(learning_rate=0.01)
lr.initialize_parameters(X_train.shape[1])

print("w:", lr.w)
print("b:", lr.b)

# Test forward
y_pred = lr.forward(X_train)
print("y_pred[:5]:", y_pred[:5])
print("Any NaN in y_pred:", np.isnan(y_pred).any())

# Test cost
cost = lr.compute_cost(y_pred, y_train)
print("Initial cost:", cost)

# Test backward
dw, db = lr.backward(X_train, y_pred, y_train)
print("dw:", dw)
print("db:", db)
print("Any NaN in y_train:", np.isnan(y_train).any())
print("y_train[:5]:", y_train[:5])
print("Total NaN in y_train:", np.isnan(y_train).sum())
print("Where:", np.where(np.isnan(y_train)))

w: [0.]
b: 0
y_pred[:5]: [0. 0. 0. 0. 0.]
Any NaN in y_pred: False
Initial cost: 1670.0624130893364
dw: [-28.95283303]
db: -49.939869170457804
Any NaN in y_train: False
y_train[:5]: [21.54945196 47.46446305 17.21865634 36.58639803 87.28898389]
Total NaN in y_train: 0
Where: (array([], dtype=int64),)


In [152]:
lr = LinearRegression(learning_rate=0.001)
lr.fit(X_train, y_train, iterations=1000)

Iteration 0: Cost = 1670.0624
Iteration 100: Cost = 1367.9081
Iteration 200: Cost = 1120.5497
Iteration 300: Cost = 918.0501
Iteration 400: Cost = 752.2741
Iteration 500: Cost = 616.5617
Iteration 600: Cost = 505.4609
Iteration 700: Cost = 414.5084
Iteration 800: Cost = 340.0502
Iteration 900: Cost = 279.0951


In [153]:
print(lr.cost_history[:10])

[np.float64(1670.0624130893364), np.float64(1666.731822144428), np.float64(1663.4078890508183), np.float64(1660.0906004994629), np.float64(1656.7799432079216), np.float64(1653.475903920306), np.float64(1650.1784694072267), np.float64(1646.8876264657383), np.float64(1643.6033619192904), np.float64(1640.325662617671)]


In [154]:
lr.save_model('model.pkl')

Model saved to model.pkl


# Evaluation



### 1. Mean Squared Error (MSE)

**Formula:**
$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_{\text{true}_i} - y_{\text{pred}_i})^2
$$

**Description:**
 - It takes the difference between each predicted value and the actual
   value, squares it (to make all differences positive and penalize
   large errors more), and then averages all of them.

**Interpretation:**
-  Lower is better. A MSE of 9.43 means the average squared difference
   between our predictions and actual values is 9.43. The squaring makes
   it hard to interpret directly which is why we use RMSE.

### 2. Root Mean Squared Error (RMSE)

**Formula:**
$$
\text{RMSE} = \sqrt{\text{MSE}}
$$

**Description:**
- RMSE is simply the square root of MSE. We take the square root to
  bring the error back to the same unit as our original data, making
  it much easier to understand.
**Interpretation:**
- Lower is better. Our RMSE of 3.07 means that on average our model's
  predictions are off by about 3 units from the actual value. This is
  much easier to understand than MSE — if we are predicting house prices
  in dollars, an RMSE of 3.07 means we are wrong by about $3 on average.


### 3. R-squared ($R^2$)

**Formula:**
$$
R^2 = 1 - \frac{\text{SSR}}{\text{SST}}
$$

**Description:**
 R² tells us how well our model fits the data overall. It compares
 our model's predictions against a very simple baseline — what if we
 just predicted the average value every single time? R² measures how
 much better our model is compared to that baseline.

**Interpretation:**
 Ranges from 0 to 1. Closer to 1 is better.
- R² = 1.0 means the model predicts perfectly
- R² = 0.0 means the model is no better than just guessing the average
- Our R² of 0.9888 (after improving in the last section) means the model explains 98.88% of the variation
  in the data which is excellent.


In [155]:
class RegressionMetrics:

    @staticmethod
    def mean_squared_error(y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    @staticmethod
    def root_mean_squared_error(y_true, y_pred):
        return np.sqrt(RegressionMetrics.mean_squared_error(y_true, y_pred))

    @staticmethod
    def r_squared(y_true, y_pred):
        # SSR = sum of squared residuals
        # SST = total sum of squares
        ssr = np.sum((y_true - y_pred) ** 2)
        sst = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (ssr / sst)

In [156]:
y_pred = lr.predict(X_test)

mse_value = RegressionMetrics.mean_squared_error(y_test, y_pred)
rmse_value = RegressionMetrics.root_mean_squared_error(y_test, y_pred)
r_squared_value = RegressionMetrics.r_squared(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse_value:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_value:.4f}")
print(f"R-squared: {r_squared_value:.4f}")

Mean Squared Error (MSE): 489.7773
Root Mean Squared Error (RMSE): 22.1309
R-squared: 0.4185


We can improve the model swalpa. We are only standardizing x but we can also standardize y.

In [158]:
train_data = pd.read_csv('/kaggle/input/datasets/andonians/random-linear-regression/train.csv').dropna()
test_data = pd.read_csv('/kaggle/input/datasets/andonians/random-linear-regression/test.csv')

x_tr = train_data['x'].values.astype(float)
y_tr = train_data['y'].values.astype(float)
x_te = test_data['x'].values.astype(float)
y_te = test_data['y'].values.astype(float)
mean_x, std_x = x_tr.mean(), x_tr.std()
mean_y, std_y = y_tr.mean(), y_tr.std()
X_train2 = ((x_tr - mean_x) / std_x).reshape(-1, 1)
X_test2  = ((x_te - mean_x) / std_x).reshape(-1, 1)
y_train2 = (y_tr - mean_y) / std_y

lr2 = LinearRegression(learning_rate=0.01)
lr2.fit(X_train2, y_train2, iterations=1000, plot_cost=False)

# Predict and convert back to original scale ( This is claude i don't fully get the syntax here)
y_pred2 = (lr2.predict(X_test2) * std_y) + mean_y

print(f"MSE:  {RegressionMetrics.mean_squared_error(y_te, y_pred2):.4f}")
print(f"RMSE: {RegressionMetrics.root_mean_squared_error(y_te, y_pred2):.4f}")
print(f"R²:   {RegressionMetrics.r_squared(y_te, y_pred2):.4f}")

Iteration 0: Cost = 0.5000
Iteration 100: Cost = 0.0710
Iteration 200: Cost = 0.0135
Iteration 300: Cost = 0.0058
Iteration 400: Cost = 0.0048
Iteration 500: Cost = 0.0047
Iteration 600: Cost = 0.0047
Iteration 700: Cost = 0.0046
Iteration 800: Cost = 0.0046
Iteration 900: Cost = 0.0046
MSE:  9.4339
RMSE: 3.0715
R²:   0.9888
